# 02 — Feature Extraction: Google AlphaEarth Embeddings

This notebook explains the feature extraction pipeline that pairs **Google AlphaEarth satellite embeddings** with **ALS canopy height labels** at 30m resolution.

## What are Google AlphaEarth Embeddings?

Google's AlphaEarth Satellite Embedding V1 provides a 64-dimensional feature vector for every 30m pixel on Earth, derived from a foundation model trained on satellite imagery. These embeddings encode spectral, spatial, and temporal patterns relevant to land surface characterization — without requiring users to design features manually.

**Key properties:**
- **64 bands** (A00–A63) per pixel
- **30m resolution**, annual composites
- Available via Google Earth Engine: `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`

## Pipeline Overview

```
For each ALS transect (487 total):
  1. Load 1m CHM GeoTIFF
  2. Resample to 30m (mean aggregation over 30×30 pixel blocks)
  3. Project 30m pixel centers to WGS84 (lon/lat)
  4. Query GEE for embedding values at those locations
  5. Join: each row = [lon, lat, A00..A63, chm, transect]

Output: embeddings_chm_30m.parquet (~1.5M rows)
```

In [ ]:
import sys
sys.path.insert(0, "..")
from src.config import DATA_DIR, EMBEDDING_COLS, SCALE, EMBEDDING_YEAR
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the extracted embeddings (pre-computed)
emb_path = DATA_DIR / "embeddings_chm_30m.parquet"
print(f"Looking for: {emb_path}")
print(f"Exists: {emb_path.exists()}")

if emb_path.exists():
    df = pd.read_parquet(emb_path)
    print(f"\nLoaded {len(df):,} samples from {df['transect'].nunique()} transects")
    print(f"Columns: {list(df.columns)}")
    print(f"\nCHM stats:")
    print(df["chm"].describe())
else:
    print("Run `make extract-embeddings` first to generate this file.")
    df = None

## Resampling Strategy: 1m → 30m

The ALS CHM GeoTIFFs are at 1m resolution. We resample to 30m using **mean aggregation** — averaging the 900 native pixels within each 30m block. This:

- Matches the Google embedding resolution (30m)
- Smooths out individual-tree noise, giving a plot-level canopy height estimate
- Reduces the dataset from ~billions of 1m pixels to ~1.5M training samples

The `coarsen(x=30, y=30).mean()` operation in xarray handles this efficiently without loading the full raster into memory.

In [ ]:
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # CHM distribution
    axes[0].hist(df["chm"], bins=60, color="forestgreen", alpha=0.7, edgecolor="black", linewidth=0.3)
    axes[0].set_xlabel("Canopy Height (m)")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"CHM Distribution at 30m (N={len(df):,})")
    axes[0].axvline(df["chm"].median(), color="red", linestyle="--", label=f"Median: {df['chm'].median():.1f}m")
    axes[0].legend()

    # Samples per transect
    counts = df.groupby("transect").size().sort_values(ascending=False)
    axes[1].bar(range(len(counts)), counts.values, color="steelblue", width=1.0)
    axes[1].set_xlabel("Transect (ranked)")
    axes[1].set_ylabel("Number of 30m samples")
    axes[1].set_title(f"Samples per Transect ({len(counts)} transects)")

    plt.tight_layout()
    plt.show()

## Embedding Space Exploration

The 64 embedding dimensions encode different aspects of the land surface. Let's visualize the first two principal components colored by canopy height to confirm that the embeddings carry vegetation structure information.

In [ ]:
if df is not None:
    from sklearn.decomposition import PCA

    # Subsample for speed
    sub = df.sample(min(50000, len(df)), random_state=42)
    X = sub[EMBEDDING_COLS].values

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=sub["chm"].values,
                    cmap="YlGn", s=1, alpha=0.3, vmin=0, vmax=40)
    plt.colorbar(sc, label="CHM (m)")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
    ax.set_title("Google AlphaEarth Embeddings — PCA colored by canopy height")
    plt.tight_layout()
    plt.show()

    print(f"Total variance explained by PC1+PC2: {pca.explained_variance_ratio_[:2].sum():.1%}")

## Key Takeaways

1. **The embeddings clearly separate low and tall canopy** — even the first two PCA components show a gradient from bare/savanna to tall forest.

2. **The dataset is imbalanced** — most samples fall in the 5–20m range (secondary forest / degraded cerrado), with fewer samples at the extremes (bare ground and tall primary forest).

3. **Parallelized GEE extraction** using 8 threads reduces total processing time from ~24 hours to ~3 hours for all 487 transects.

4. **Incremental saving** (every 20 transects) makes the pipeline robust to GEE timeouts or network interruptions.